In [1]:
import stan
import pandas as pd
import numpy as np
import nest_asyncio
import arviz as az
nest_asyncio.apply()

# ===================================================================
# Hybrid model from Ott's (Benchmark)
data = pd.read_csv("../../data/raw_data/all_participants_data.csv", on_bad_lines='skip')
data = data.dropna()

m03 = '''
data {
  int<lower=0> N;
  int<lower=0,upper=1> response[N];

  vector[N] dv;

  vector[N] is_basic_1;
  vector[N] is_basic_2;
  vector[N] is_basic_3;
  vector[N] is_basic_4;

  vector[N] is_23;
  vector[N] is_14;

  vector[N] is_full_energy;
  vector[N] is_low_energy_LC;
  vector[N] is_low_energy_HC;

  int<lower=1> N_subjects;
  int<lower=1,upper=N_subjects> vpn[N];
}

parameters {
  // hyperparameters
  real mu_theta_basic_1;
  real mu_theta_basic_2;
  real mu_theta_basic_3;
  real mu_theta_basic_4;

  real mu_beta_dv_23;
  real mu_beta_dv_14;

  real<lower=0> sigma_theta_basic_1;
  real<lower=0> sigma_theta_basic_2;
  real<lower=0> sigma_theta_basic_3;
  real<lower=0> sigma_theta_basic_4;

  real<lower=0> sigma_beta_dv_23;
  real<lower=0> sigma_beta_dv_14;

  // subject-level parameters
  vector[N_subjects] theta_basic_1;
  vector[N_subjects] theta_basic_2;
  vector[N_subjects] theta_basic_3;
  vector[N_subjects] theta_basic_4;

  vector[N_subjects] beta_dv_23;
  vector[N_subjects] beta_dv_14;

  // fixed effects
  real theta_full_energy;
  real theta_low_energy_LC;
  real theta_low_energy_HC;
}

model {
  vector[N] eta;

  // hyperpriors
  mu_theta_basic_1 ~ normal(0,2);
  mu_theta_basic_2 ~ normal(0,2);
  mu_theta_basic_3 ~ normal(0,2);
  mu_theta_basic_4 ~ normal(0,2);

  mu_beta_dv_23 ~ normal(0,2);
  mu_beta_dv_14 ~ normal(0,2);

  sigma_theta_basic_1 ~ normal(0,2);
  sigma_theta_basic_2 ~ normal(0,2);
  sigma_theta_basic_3 ~ normal(0,2);
  sigma_theta_basic_4 ~ normal(0,2);

  sigma_beta_dv_23 ~ normal(0,2);
  sigma_beta_dv_14 ~ normal(0,2);

  // hierarchical priors
  theta_basic_1 ~ normal(mu_theta_basic_1, sigma_theta_basic_1);
  theta_basic_2 ~ normal(mu_theta_basic_2, sigma_theta_basic_2);
  theta_basic_3 ~ normal(mu_theta_basic_3, sigma_theta_basic_3);
  theta_basic_4 ~ normal(mu_theta_basic_4, sigma_theta_basic_4);

  beta_dv_23 ~ normal(mu_beta_dv_23, sigma_beta_dv_23);
  beta_dv_14 ~ normal(mu_beta_dv_14, sigma_beta_dv_14);

  // fixed effects priors
  theta_full_energy ~ normal(0,2);
  theta_low_energy_LC ~ normal(0,2);
  theta_low_energy_HC ~ normal(0,2);

  // linear predictor
  for (n in 1:N) {
    eta[n] =
        theta_full_energy * is_full_energy[n]
      + theta_low_energy_LC * is_low_energy_LC[n]
      + theta_low_energy_HC * is_low_energy_HC[n]

      + theta_basic_1[vpn[n]] * is_basic_1[n]
      + theta_basic_2[vpn[n]] * is_basic_2[n]
      + theta_basic_3[vpn[n]] * is_basic_3[n]
      + theta_basic_4[vpn[n]] * is_basic_4[n]

      + beta_dv_23[vpn[n]] * dv[n] * is_23[n]
      + beta_dv_14[vpn[n]] * dv[n] * is_14[n];
  }

  // likelihood
  response ~ bernoulli_logit(eta);
}

generated quantities {
  vector[N] log_lik;
  vector[N] response_new;

  vector[N_subjects] theta_basic_1_rep;
  vector[N_subjects] theta_basic_2_rep;
  vector[N_subjects] theta_basic_3_rep;
  vector[N_subjects] theta_basic_4_rep;

  vector[N_subjects] beta_dv_rep_23;
  vector[N_subjects] beta_dv_rep_14;

  // log likelihood
  for (n in 1:N) {
    log_lik[n] = bernoulli_logit_lpmf(response[n] | 
        theta_full_energy * is_full_energy[n]
      + theta_low_energy_LC * is_low_energy_LC[n]
      + theta_low_energy_HC * is_low_energy_HC[n]
      + theta_basic_1[vpn[n]] * is_basic_1[n]
      + theta_basic_2[vpn[n]] * is_basic_2[n]
      + theta_basic_3[vpn[n]] * is_basic_3[n]
      + theta_basic_4[vpn[n]] * is_basic_4[n]
      + beta_dv_23[vpn[n]] * dv[n] * is_23[n]
      + beta_dv_14[vpn[n]] * dv[n] * is_14[n]
    );
  }

  // simulate new subject-level parameters
  for (j in 1:N_subjects) {
    theta_basic_1_rep[j] = normal_rng(mu_theta_basic_1, sigma_theta_basic_1);
    theta_basic_2_rep[j] = normal_rng(mu_theta_basic_2, sigma_theta_basic_2);
    theta_basic_3_rep[j] = normal_rng(mu_theta_basic_3, sigma_theta_basic_3);
    theta_basic_4_rep[j] = normal_rng(mu_theta_basic_4, sigma_theta_basic_4);

    beta_dv_rep_23[j] = normal_rng(mu_beta_dv_23, sigma_beta_dv_23);
    beta_dv_rep_14[j] = normal_rng(mu_beta_dv_14, sigma_beta_dv_14);
  }

  // posterior predictive
  for (n in 1:N) {
    response_new[n] = bernoulli_logit_rng(
        theta_full_energy * is_full_energy[n]
      + theta_low_energy_LC * is_low_energy_LC[n]
      + theta_low_energy_HC * is_low_energy_HC[n]
      + theta_basic_1_rep[vpn[n]] * is_basic_1[n]
      + theta_basic_2_rep[vpn[n]] * is_basic_2[n]
      + theta_basic_3_rep[vpn[n]] * is_basic_3[n]
      + theta_basic_4_rep[vpn[n]] * is_basic_4[n]
      + beta_dv_rep_23[vpn[n]] * dv[n] * is_23[n]
      + beta_dv_rep_14[vpn[n]] * dv[n] * is_14[n]
    );
  }
}
'''




idx = (data['timeout'] == 0)
response = (data.loc[idx,['response']] == 0).to_numpy(dtype='int').squeeze()
is_full_energy = data.loc[idx,['is_full_energy']].to_numpy(dtype='int').squeeze()
is_low_energy_LC = data.loc[idx,['is_low_energy_LC']].to_numpy(dtype='int').squeeze()
is_low_energy_HC = data.loc[idx,['is_low_energy_HC']].to_numpy(dtype='int').squeeze()
is_basic_1 = data.loc[idx,['is_basic_1']].to_numpy(dtype='int').squeeze()
is_basic_2 = data.loc[idx,['is_basic_2']].to_numpy(dtype='int').squeeze()
is_basic_3 = data.loc[idx,['is_basic_3']].to_numpy(dtype='int').squeeze()
is_basic_4 = data.loc[idx,['is_basic_4']].to_numpy(dtype='int').squeeze()
is_14 = data.loc[idx,['is_14']].to_numpy(dtype='int').squeeze()
is_23 = data.loc[idx,['is_23']].to_numpy(dtype='int').squeeze()
vpn = data.loc[idx, ['vpn']].to_numpy().squeeze()
vpn = vpn - vpn.min() + 1
N_subjects = vpn.max()
dv = data.loc[idx,['dv_planning']].to_numpy().squeeze()

dat_dict  = {'N':len(response),         
            'response':response,
            'dv':dv,      
            'is_full_energy':is_full_energy ,
            'is_low_energy_LC':is_low_energy_LC,
            'is_low_energy_HC':is_low_energy_HC,
            'is_basic_1':is_basic_1,
            'is_basic_2':is_basic_2,
            'is_basic_3':is_basic_3,
            'is_basic_4':is_basic_4,
            'is_23':is_23,
            'is_14':is_14,
            'N_subjects':N_subjects,
            'vpn':vpn
            } 

sm03 = stan.build(m03, data=dat_dict)

res_sm03 = sm03.sample(
    num_chains=4,
    num_samples=4000,
    num_warmup=4000,
    adapt_delta=0.99,
    random_seed=101
    );


idata = az.from_pystan(res_sm03, log_likelihood='log_lik', posterior_predictive='response_new')

ModuleNotFoundError: No module named 'pkg_resources'